In [ ]:
import random
import requests
from datetime import datetime, timedelta

# Configuración de CouchDB
COUCHDB_URL = "http://127.0.0.1:5984/_utils/#database/mercadotico360/_all_docs"
AUTH = ("Rojaja15", "Javier15")
BATCH_SIZE = 5000 

# Define las 8 categorías exigidas con sus atributos variables
CATEGORIAS = {
    "Alimentos": ["origen", "tueste", "fecha_vencimiento", "registro_sanitario"],
    "Tecnología": ["marca", "garantia_meses", "resolucion", "voltaje"],
    "Ropa": ["talla", "color", "material", "genero"],
    "Artesanías": ["material", "tecnica", "region_origen", "hecho_a_mano"],
    "Servicios": ["duracion_horas", "modalidad", "incluye_certificado"],
    "Hogar": ["dimensiones", "peso_kg", "color", "ensamblado_requerido"],
    "Deportes": ["disciplina", "resistencia_agua", "peso_g"],
    "Belleza": ["tipo_piel", "contenido_ml", "libre_crueldad"]
}

def enviar_lote(documentos):
    res = requests.post(COUCHDB_URL, json={"docs": documentos}, headers=HEADERS)
    if res.status_code not in [200, 201]:
        print(f"Error al enviar lote: {res.status_code}")

print("--- Iniciando Generación y Carga Masiva en CouchDB ---")

# 1. GENERAR Y CARGAR 25,000 PRODUCTOS
print("Generando 25,000 productos distribuidos en 8 categorías...")
docs_lote = []
cats_keys = list(CATEGORIAS.keys())

for i in range(1, 25001):
    cat = cats_keys[i % len(cats_keys)]
    prod = {
        "_id": f"prod_{i}",
        "tipo": "producto",
        "sku": f"SKU-{cat[:3].upper()}-{i:05d}",
        "nombre": f"Producto {cat} #{i}",
        "categoria": cat,
        "precio": round(random.uniform(1000.0, 250000.0), 2),
        "stock": random.randint(5, 500),
        "atributos": {attr: f"Valor_{attr}_{i}" for attr in CATEGORIAS[cat]}
    }
    docs_lote.append(prod)
    if len(docs_lote) >= BATCH_SIZE:
        enviar_lote(docs_lote)
        docs_lote = []
if docs_lote:
    enviar_lote(docs_lote)
    docs_lote = []
print("✔ 25,000 Productos cargados exitosamente.")

# 2. GENERAR Y CARGAR 10,000 CLIENTES
print("Generando 10,000 clientes...")
for i in range(1, 10001):
    cli = {
        "_id": f"cli_{i}",
        "tipo": "cliente",
        "nombre": f"Cliente Costa Rica #{i}",
        "email": f"usuario{i}@mercadotico.cr",
        "provincia": random.choice(["San José", "Alajuela", "Cartago", "Heredia", "Guanacaste", "Puntarenas", "Limón"])
    }
    docs_lote.append(cli)
    if len(docs_lote) >= BATCH_SIZE:
        enviar_lote(docs_lote)
        docs_lote = []
if docs_lote:
    enviar_lote(docs_lote)
    docs_lote = []
print("✔ 10,000 Clientes cargados exitosamente.")

# 3. GENERAR Y CARGAR 50,000 PEDIDOS CON SNAPSHOTS DESNORMALIZADOS
print("Generando 50,000 pedidos con desnormalización e histórico...")
fecha_base = datetime.now()

for i in range(1, 50001):
    num_lineas = random.randint(1, 4)
    lineas = []
    monto_total = 0.0

    for _ in range(num_lineas):
        prod_id_num = random.randint(1, 25000)
        precio_unit = round(random.uniform(2000.0, 50000.0), 2)
        cant = random.randint(1, 3)
        monto_total += precio_unit * cant

        # Snapshot para reconstruir el producto histórico aunque luego cambie
        lineas.append({
            "producto_id": f"prod_{prod_id_num}",
            "nombre_snapshot": f"Producto Histórico Snapshot #{prod_id_num}",
            "precio_unitario": precio_unit,
            "cantidad": cant
        })

    fecha_pedido = (fecha_base - timedelta(days=random.randint(0, 180))).isoformat()

    ped = {
        "_id": f"ped_{i}",
        "tipo": "pedido",
        "cliente_id": f"cli_{random.randint(1, 10000)}",
        "fecha": fecha_pedido,
        "estado": random.choice(["COMPLETADO", "PROCESANDO", "ENVIADO", "CANCELADO"]),
        "total": round(monto_total, 2),
        "lineas_detalle": lineas
    }
    docs_lote.append(ped)
    if len(docs_lote) >= BATCH_SIZE:
        enviar_lote(docs_lote)
        docs_lote = []
if docs_lote:
    enviar_lote(docs_lote)
    docs_lote = []

print("✔ 50,000 Pedidos cargados exitosamente.")
print("--- Carga masiva finalizada con éxito ---")

--- Iniciando Generación y Carga Masiva en CouchDB ---
Generando 25,000 productos distribuidos en 8 categorías...
Error al enviar lote: 401
Error al enviar lote: 401
Error al enviar lote: 401
Error al enviar lote: 401
Error al enviar lote: 401
✔ 25,000 Productos cargados exitosamente.
Generando 10,000 clientes...
Error al enviar lote: 401
Error al enviar lote: 401
✔ 10,000 Clientes cargados exitosamente.
Generando 50,000 pedidos con desnormalización e histórico...
Error al enviar lote: 401
Error al enviar lote: 401
Error al enviar lote: 401
Error al enviar lote: 401
Error al enviar lote: 401
Error al enviar lote: 401
Error al enviar lote: 401
Error al enviar lote: 401
Error al enviar lote: 401
Error al enviar lote: 401
✔ 50,000 Pedidos cargados exitosamente.
--- Carga masiva finalizada con éxito ---


In [4]:
import requests
import random
import uuid

COUCHDB_URL = "http://127.0.0.1:5984/_utils/#database/mercadotico360/_all_docs"
AUTH = ("Rojaja15", "Javier15")
BATCH_SIZE = 5000

CATEGORIAS = [
    "alimentos", "ropa", "tecnologia", "artesanias", 
    "servicios", "hogar", "deportes", "belleza"
]

def generar_producto(i):
    cat = random.choice(CATEGORIAS)
    doc = {
        "_id": f"prod:{i}",
        "tipo": "producto",
        "nombre": f"Producto MercadoTico {i}",
        "categoria": cat,
        "precio": round(random.uniform(2000, 150000), 2),
        "comercio_id": f"comercio_{random.randint(1, 100)}"
    }
    # Atributos variables según la categoría (Demuestra esquema flexible)
    if cat == "alimentos":
        doc["atributos"] = {"fecha_vencimiento": "2026-12-31", "perecedero": True}
    elif cat == "ropa":
        doc["atributos"] = {"talla": random.choice(["S", "M", "L"]), "color": "Azul"}
    elif cat == "tecnologia":
        doc["atributos"] = {"garantia_meses": 12, "voltaje": "110V"}
    else:
        doc["atributos"] = {"origen": "Costa Rica", "artesanal": True}
    return doc

def generar_cliente(i):
    return {
        "_id": f"cli:{i}",
        "tipo": "cliente",
        "nombre": f"Cliente {i}",
        "correo": f"usuario{i}@correo.cr",
        "provincia": random.choice(["San José", "Alajuela", "Cartago", "Heredia"])
    }

def generar_pedido(i):
    prod_id = random.randint(1, 25000)
    cant = random.randint(1, 3)
    precio_unit = round(random.uniform(5000, 20000), 2)
    
    return {
        "_id": f"ped:{i}",
        "tipo": "pedido",
        "cliente_id": f"cli:{random.randint(1, 10000)}",
        "fecha": "2026-09-23T10:00:00Z",
        "estado": "completado",
        # Información embebida para congelar el historial del pedido
        "lineas_detalle": [
            {
                "producto_id": f"prod:{prod_id}",
                "nombre_historico": f"Producto MercadoTico {prod_id}",
                "cantidad": cant,
                "precio_unitario": precio_unit
            }
        ],
        "monto_total": round(cant * precio_unit, 2)
    }

def enviar_lote(documentos):
    headers = {"Content-Type": "application/json"}
    payload = {"docs": documentos}
    res = requests.post(COUCHDB_URL, json=payload, auth=AUTH, headers=headers)
    if res.status_status_code not in [200, 201]:
        print(f"Error al cargar lote: {res.status_code}")

def main():
    lote = []
    total = 0

    print("Cargando 25.000 Productos...")
    for i in range(1, 25001):
        lote.append(generar_producto(i))
        if len(lote) >= BATCH_SIZE:
            enviar_lote(lote)
            total += len(lote)
            print(f"Insertados: {total}/85000")
            lote = []

    print("Cargando 10.000 Clientes...")
    for i in range(1, 10001):
        lote.append(generar_cliente(i))
        if len(lote) >= BATCH_SIZE:
            enviar_lote(lote)
            total += len(lote)
            print(f"Insertados: {total}/85000")
            lote = []

    print("Cargando 50.000 Pedidos...")
    for i in range(1, 50001):
        lote.append(generar_pedido(i))
        if len(lote) >= BATCH_SIZE:
            enviar_lote(lote)
            total += len(lote)
            print(f"Insertados: {total}/85000")
            lote = []

    if lote:
        enviar_lote(lote)
        total += len(lote)

    print(f" Carga completa. Total de documentos insertados: {total}")

if __name__ == "__main__":
    main()

Cargando 25.000 Productos...


AttributeError: 'Response' object has no attribute 'status_status_code'

In [5]:
import requests
import random
import uuid

COUCHDB_URL = "http://127.0.0.1:5984/_utils/#database/mercadotico360/_all_docs"
AUTH = ("Rojaja15", "Javier15")
BATCH_SIZE = 5000

CATEGORIAS = [
    "alimentos", "ropa", "tecnologia", "artesanias", 
    "servicios", "hogar", "deportes", "belleza"
]

def generar_producto(i):
    cat = random.choice(CATEGORIAS)
    doc = {
        "_id": f"prod:{i}",
        "tipo": "producto",
        "nombre": f"Producto MercadoTico {i}",
        "categoria": cat,
        "precio": round(random.uniform(2000, 150000), 2),
        "comercio_id": f"comercio_{random.randint(1, 100)}"
    }
    # Atributos variables según la categoría (Demuestra esquema flexible)
    if cat == "alimentos":
        doc["atributos"] = {"fecha_vencimiento": "2026-12-31", "perecedero": True}
    elif cat == "ropa":
        doc["atributos"] = {"talla": random.choice(["S", "M", "L"]), "color": "Azul"}
    elif cat == "tecnologia":
        doc["atributos"] = {"garantia_meses": 12, "voltaje": "110V"}
    else:
        doc["atributos"] = {"origen": "Costa Rica", "artesanal": True}
    return doc

def generar_cliente(i):
    return {
        "_id": f"cli:{i}",
        "tipo": "cliente",
        "nombre": f"Cliente {i}",
        "correo": f"usuario{i}@correo.cr",
        "provincia": random.choice(["San José", "Alajuela", "Cartago", "Heredia"])
    }

def generar_pedido(i):
    prod_id = random.randint(1, 25000)
    cant = random.randint(1, 3)
    precio_unit = round(random.uniform(5000, 20000), 2)
    
    return {
        "_id": f"ped:{i}",
        "tipo": "pedido",
        "cliente_id": f"cli:{random.randint(1, 10000)}",
        "fecha": "2026-09-23T10:00:00Z",
        "estado": "completado",
        # Información embebida para congelar el historial del pedido
        "lineas_detalle": [
            {
                "producto_id": f"prod:{prod_id}",
                "nombre_historico": f"Producto MercadoTico {prod_id}",
                "cantidad": cant,
                "precio_unitario": precio_unit
            }
        ],
        "monto_total": round(cant * precio_unit, 2)
    }

def enviar_lote(documentos):
    headers = {"Content-Type": "application/json"}
    payload = {"docs": documentos}
    res = requests.post(COUCHDB_URL, json=payload, auth=AUTH, headers=headers)
    if res.status_status_code not in [200, 201]:
        print(f"Error al cargar lote: {res.status_code}")

def main():
    lote = []
    total = 0

    print("Cargando 25.000 Productos...")
    for i in range(1, 25001):
        lote.append(generar_producto(i))
        if len(lote) >= BATCH_SIZE:
            enviar_lote(lote)
            total += len(lote)
            print(f"Insertados: {total}/85000")
            lote = []

    print("Cargando 10.000 Clientes...")
    for i in range(1, 10001):
        lote.append(generar_cliente(i))
        if len(lote) >= BATCH_SIZE:
            enviar_lote(lote)
            total += len(lote)
            print(f"Insertados: {total}/85000")
            lote = []

    print("Cargando 50.000 Pedidos...")
    for i in range(1, 50001):
        lote.append(generar_pedido(i))
        if len(lote) >= BATCH_SIZE:
            enviar_lote(lote)
            total += len(lote)
            print(f"Insertados: {total}/85000")
            lote = []

    if lote:
        enviar_lote(lote)
        total += len(lote)

    print(f" Carga completa. Total de documentos insertados: {total}")

if __name__ == "__main__":
    main()

Cargando 25.000 Productos...


AttributeError: 'Response' object has no attribute 'status_status_code'

In [6]:
import requests
import random
import uuid

COUCHDB_URL = "http://127.0.0.1:5984/_utils/#database/mercadotico360/_all_docs"
AUTH = ("Rojaja15", "Javier15")
BATCH_SIZE = 5000

CATEGORIAS = [
    "alimentos", "ropa", "tecnologia", "artesanias", 
    "servicios", "hogar", "deportes", "belleza"
]

def generar_producto(i):
    cat = random.choice(CATEGORIAS)
    doc = {
        "_id": f"prod:{i}",
        "tipo": "producto",
        "nombre": f"Producto MercadoTico {i}",
        "categoria": cat,
        "precio": round(random.uniform(2000, 150000), 2),
        "comercio_id": f"comercio_{random.randint(1, 100)}"
    }
    # Atributos variables según la categoría (Demuestra esquema flexible)
    if cat == "alimentos":
        doc["atributos"] = {"fecha_vencimiento": "2026-12-31", "perecedero": True}
    elif cat == "ropa":
        doc["atributos"] = {"talla": random.choice(["S", "M", "L"]), "color": "Azul"}
    elif cat == "tecnologia":
        doc["atributos"] = {"garantia_meses": 12, "voltaje": "110V"}
    else:
        doc["atributos"] = {"origen": "Costa Rica", "artesanal": True}
    return doc

def generar_cliente(i):
    return {
        "_id": f"cli:{i}",
        "tipo": "cliente",
        "nombre": f"Cliente {i}",
        "correo": f"usuario{i}@correo.cr",
        "provincia": random.choice(["San José", "Alajuela", "Cartago", "Heredia"])
    }

def generar_pedido(i):
    prod_id = random.randint(1, 25000)
    cant = random.randint(1, 3)
    precio_unit = round(random.uniform(5000, 20000), 2)
    
    return {
        "_id": f"ped:{i}",
        "tipo": "pedido",
        "cliente_id": f"cli:{random.randint(1, 10000)}",
        "fecha": "2026-09-23T10:00:00Z",
        "estado": "completado",
        # Información embebida para congelar el historial del pedido
        "lineas_detalle": [
            {
                "producto_id": f"prod:{prod_id}",
                "nombre_historico": f"Producto MercadoTico {prod_id}",
                "cantidad": cant,
                "precio_unitario": precio_unit
            }
        ],
        "monto_total": round(cant * precio_unit, 2)
    }

def enviar_lote(documentos):
    headers = {"Content-Type": "application/json"}
    payload = {"docs": documentos}
    res = requests.post(COUCHDB_URL, json=payload, auth=AUTH, headers=headers)
    if res.status_status_code not in [200, 201]:
        print(f"Error al cargar lote: {res.status_code}")

def main():
    lote = []
    total = 0

    print("Cargando 25.000 Productos...")
    for i in range(1, 25001):
        lote.append(generar_producto(i))
        if len(lote) >= BATCH_SIZE:
            enviar_lote(lote)
            total += len(lote)
            print(f"Insertados: {total}/85000")
            lote = []

    print("Cargando 10.000 Clientes...")
    for i in range(1, 10001):
        lote.append(generar_cliente(i))
        if len(lote) >= BATCH_SIZE:
            enviar_lote(lote)
            total += len(lote)
            print(f"Insertados: {total}/85000")
            lote = []

    print("Cargando 50.000 Pedidos...")
    for i in range(1, 50001):
        lote.append(generar_pedido(i))
        if len(lote) >= BATCH_SIZE:
            enviar_lote(lote)
            total += len(lote)
            print(f"Insertados: {total}/85000")
            lote = []

    if lote:
        enviar_lote(lote)
        total += len(lote)

    print(f" Carga completa. Total de documentos insertados: {total}")

if __name__ == "__main__":
    main()

Cargando 25.000 Productos...


AttributeError: 'Response' object has no attribute 'status_status_code'

In [7]:
import random
import requests

# Configuración de conexión
COUCHDB_URL = "http://127.0.0.1:5984/_utils/#database/mercadotico360/_all_docs"
AUTH = ("Rojaja15", "Javier15")
BATCH_SIZE = 5000

CATEGORIAS = [
    "alimentos",
    "ropa",
    "tecnologia",
    "artesanias",
    "servicios",
    "hogar",
    "deportes",
    "belleza",
]


def generar_producto(i):
    cat = random.choice(CATEGORIAS)
    doc = {
        "_id": f"prod:{i}",
        "tipo": "producto",
        "nombre": f"Producto MercadoTico {i}",
        "categoria": cat,
        "precio": round(random.uniform(2000, 150000), 2),
        "comercio_id": f"comercio_{random.randint(1, 100)}",
    }
    # Atributos variables por categoría (Esquema flexible)
    if cat == "alimentos":
        doc["atributos"] = {
            "fecha_vencimiento": "2026-12-31",
            "perecedero": True,
        }
    elif cat == "ropa":
        doc["atributos"] = {
            "talla": random.choice(["S", "M", "L"]),
            "color": "Azul",
        }
    elif cat == "tecnologia":
        doc["atributos"] = {"garantia_meses": 12, "voltaje": "110V"}
    else:
        doc["atributos"] = {"origen": "Costa Rica", "artesanal": True}
    return doc


def generar_cliente(i):
    return {
        "_id": f"cli:{i}",
        "tipo": "cliente",
        "nombre": f"Cliente {i}",
        "correo": f"usuario{i}@correo.cr",
        "provincia": random.choice(["San José", "Alajuela", "Cartago", "Heredia"]),
    }


def generar_pedido(i):
    prod_id = random.randint(1, 25000)
    cant = random.randint(1, 3)
    precio_unit = round(random.uniform(5000, 20000), 2)

    return {
        "_id": f"ped:{i}",
        "tipo": "pedido",
        "cliente_id": f"cli:{random.randint(1, 10000)}",
        "fecha": "2026-09-23T10:00:00Z",
        "estado": "completado",
        # Estructura anidada para conservar historial de precios y nombres
        "lineas_detalle": [
            {
                "producto_id": f"prod:{prod_id}",
                "nombre_historico": f"Producto MercadoTico {prod_id}",
                "cantidad": cant,
                "precio_unitario": precio_unit,
            }
        ],
        "monto_total": round(cant * precio_unit, 2),
    }


def enviar_lote(documentos):
    headers = {"Content-Type": "application/json"}
    payload = {"docs": documentos}
    res = requests.post(COUCHDB_URL, json=payload, auth=AUTH, headers=headers)

    # Verificación corregida usando status_code
    if res.status_code not in [200, 201]:
        print(f"Error al cargar lote: {res.status_code} - {res.text}")


def main():
    lote = []
    total = 0

    print("Cargando 25.000 Productos...")
    for i in range(1, 25001):
        lote.append(generar_producto(i))
        if len(lote) >= BATCH_SIZE:
            enviar_lote(lote)
            total += len(lote)
            print(f"Insertados: {total}/85000")
            lote = []

    print("Cargando 10.000 Clientes...")
    for i in range(1, 10001):
        lote.append(generar_cliente(i))
        if len(lote) >= BATCH_SIZE:
            enviar_lote(lote)
            total += len(lote)
            print(f"Insertados: {total}/85000")
            lote = []

    print("Cargando 50.000 Pedidos...")
    for i in range(1, 50001):
        lote.append(generar_pedido(i))
        if len(lote) >= BATCH_SIZE:
            enviar_lote(lote)
            total += len(lote)
            print(f"Insertados: {total}/85000")
            lote = []

    # Enviar residuales si quedaron elementos pendientes en el lote
    if lote:
        enviar_lote(lote)
        total += len(lote)

    print(f"Proceso finalizado. Total de documentos insertados: {total}")


if __name__ == "__main__":
    main()

Cargando 25.000 Productos...
Error al cargar lote: 405 - {"error":"method_not_allowed","reason":"Only GET,HEAD allowed"}

Insertados: 5000/85000
Error al cargar lote: 405 - {"error":"method_not_allowed","reason":"Only GET,HEAD allowed"}

Insertados: 10000/85000
Error al cargar lote: 405 - {"error":"method_not_allowed","reason":"Only GET,HEAD allowed"}

Insertados: 15000/85000
Error al cargar lote: 405 - {"error":"method_not_allowed","reason":"Only GET,HEAD allowed"}

Insertados: 20000/85000
Error al cargar lote: 405 - {"error":"method_not_allowed","reason":"Only GET,HEAD allowed"}

Insertados: 25000/85000
Cargando 10.000 Clientes...
Error al cargar lote: 405 - {"error":"method_not_allowed","reason":"Only GET,HEAD allowed"}

Insertados: 30000/85000
Error al cargar lote: 405 - {"error":"method_not_allowed","reason":"Only GET,HEAD allowed"}

Insertados: 35000/85000
Cargando 50.000 Pedidos...
Error al cargar lote: 405 - {"error":"method_not_allowed","reason":"Only GET,HEAD allowed"}

Inser

In [8]:
import random
import requests

#  CORRECTO: Endpoint de la API REST de CouchDB
COUCHDB_URL = "http://127.0.0.1:5984/mercadotico360/_bulk_docs"
AUTH = ("Rojaja15", "Javier15")
BATCH_SIZE = 5000

CATEGORIAS = [
    "alimentos",
    "ropa",
    "tecnologia",
    "artesanias",
    "servicios",
    "hogar",
    "deportes",
    "belleza",
]


def generar_producto(i):
    cat = random.choice(CATEGORIAS)
    doc = {
        "_id": f"prod:{i}",
        "tipo": "producto",
        "nombre": f"Producto MercadoTico {i}",
        "categoria": cat,
        "precio": round(random.uniform(2000, 150000), 2),
        "comercio_id": f"comercio_{random.randint(1, 100)}",
    }
    if cat == "alimentos":
        doc["atributos"] = {
            "fecha_vencimiento": "2026-12-31",
            "perecedero": True,
        }
    elif cat == "ropa":
        doc["atributos"] = {
            "talla": random.choice(["S", "M", "L"]),
            "color": "Azul",
        }
    elif cat == "tecnologia":
        doc["atributos"] = {"garantia_meses": 12, "voltaje": "110V"}
    else:
        doc["atributos"] = {"origen": "Costa Rica", "artesanal": True}
    return doc


def generar_cliente(i):
    return {
        "_id": f"cli:{i}",
        "tipo": "cliente",
        "nombre": f"Cliente {i}",
        "correo": f"usuario{i}@correo.cr",
        "provincia": random.choice(["San José", "Alajuela", "Cartago", "Heredia"]),
    }


def generar_pedido(i):
    prod_id = random.randint(1, 25000)
    cant = random.randint(1, 3)
    precio_unit = round(random.uniform(5000, 20000), 2)

    return {
        "_id": f"ped:{i}",
        "tipo": "pedido",
        "cliente_id": f"cli:{random.randint(1, 10000)}",
        "fecha": "2026-09-23T10:00:00Z",
        "estado": "completado",
        "lineas_detalle": [
            {
                "producto_id": f"prod:{prod_id}",
                "nombre_historico": f"Producto MercadoTico {prod_id}",
                "cantidad": cant,
                "precio_unitario": precio_unit,
            }
        ],
        "monto_total": round(cant * precio_unit, 2),
    }


def enviar_lote(documentos):
    headers = {"Content-Type": "application/json"}
    payload = {"docs": documentos}
    res = requests.post(COUCHDB_URL, json=payload, auth=AUTH, headers=headers)

    if res.status_code in [200, 201]:
        return True
    else:
        print(f"Error al cargar lote: {res.status_code} - {res.text}")
        return False


def main():
    lote = []
    total = 0

    print("Cargando 25.000 Productos...")
    for i in range(1, 25001):
        lote.append(generar_producto(i))
        if len(lote) >= BATCH_SIZE:
            if enviar_lote(lote):
                total += len(lote)
                print(f"Insertados exitosamente: {total}/85000")
            lote = []

    print("Cargando 10.000 Clientes...")
    for i in range(1, 10001):
        lote.append(generar_cliente(i))
        if len(lote) >= BATCH_SIZE:
            if enviar_lote(lote):
                total += len(lote)
                print(f"Insertados exitosamente: {total}/85000")
            lote = []

    print("Cargando 50.000 Pedidos...")
    for i in range(1, 50001):
        lote.append(generar_pedido(i))
        if len(lote) >= BATCH_SIZE:
            if enviar_lote(lote):
                total += len(lote)
                print(f"Insertados exitosamente: {total}/85000")
            lote = []

    if lote:
        if enviar_lote(lote):
            total += len(lote)

    print(f"\nProceso finalizado. Total de documentos insertados: {total}")


if __name__ == "__main__":
    main()

Cargando 25.000 Productos...
Insertados exitosamente: 5000/85000
Insertados exitosamente: 10000/85000
Insertados exitosamente: 15000/85000
Insertados exitosamente: 20000/85000
Insertados exitosamente: 25000/85000
Cargando 10.000 Clientes...
Insertados exitosamente: 30000/85000
Insertados exitosamente: 35000/85000
Cargando 50.000 Pedidos...
Insertados exitosamente: 40000/85000
Insertados exitosamente: 45000/85000
Insertados exitosamente: 50000/85000
Insertados exitosamente: 55000/85000
Insertados exitosamente: 60000/85000
Insertados exitosamente: 65000/85000
Insertados exitosamente: 70000/85000
Insertados exitosamente: 75000/85000
Insertados exitosamente: 80000/85000
Insertados exitosamente: 85000/85000

Proceso finalizado. Total de documentos insertados: 85000
